# Лабораторная работа №1. Нормализация
Выполнила: Перминова А.А., группа М4221

## Разведочный анализ корпуса (EDA)

In [99]:
import csv
import os
import re
import unicodedata
from collections import Counter

import pandas as pd

In [100]:
INPUT_PATH = "data/RUSLAN/metadata_RUSLAN_22200.csv"
OUT_DIR = "data/eda"
CSV_KWARGS = {"sep": "|", "quoting": csv.QUOTE_NONE}

# типы ненормализованных данных
PATTERNS = {
    "цифры": re.compile(r"\d"),
    "латиница": re.compile(r"[A-Za-z]"),
    "технические символы": re.compile(r"[*/@+<>\\|^~_#={}]"),
    "обозначения": re.compile(r"[%°$€₽№§©®&]"),
    "сокращения: единицы измерения": re.compile(r"(?<= )(?:мм|см|дм|км|мг|кг|мл|л|мин|сек)(?![А-Яа-яЁё])"),
    "сокращения: деньги": re.compile(
        r"(?<![А-Яа-яЁё])(?:руб\.|коп\.|тыс\.|млн\.|млрд\.)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: географические": re.compile(
        r"(?<![А-Яа-яЁё])(?:ул\.|пл\.|просп\.|наб\.|пер\.|бул\.|ш\.|тр\.|кв\.|обл\.|р-н|пос\.|дер\.)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: научные и учебные": re.compile(
        r"(?<![А-Яа-яЁё])(?:рис\.|табл\.|гл\.|стр\.|см\.|ср\.|напр\.|прим\.|разд\.|парагр\.|изд\.|ред\.|сост\.)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: должности и звания": re.compile(
        r"(?<![А-Яа-яЁё])(?:проф\.|акад\.|доц\.|ассист\.|инж\.|техн\.|дир\.|зав\.|зам\.|тов\.|гр\.|г-н|г-жа)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: количество": re.compile(
        r"(?<![А-Яа-яЁё])(?:чел\.|экз\.|шт\.|ед\.)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: общеупотребительные": re.compile(
        r"(?<![А-Яа-яЁё])(?:т\.\s*[дпек]\.|др\.|им\.)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: официальные": re.compile(
        r"(?<![А-Яа-яЁё])(?:гос\.|гос-во|об-во|орг\.|орг-ция|предпр\.|учр\.|стр-во|хоз-во)(?![А-Яа-яЁё])",
        re.IGNORECASE
    ),
    
    "сокращения: микс": re.compile(
        r"(?<![А-Яа-яЁё])(?:г\.?|в\.|вв\.|н\.\s*э\.|ч\.?)(?![А-Яа-яЁё])"),
    "инициалы": re.compile(
        r"(?<![А-Яа-яЁё])"
        r"(?:[А-ЯЁ]\.\s*){1,2}[А-ЯЁ][а-яё]+"
        r"|"
        r"[А-ЯЁ][а-яё]+\s+(?:[А-ЯЁ]\.\s*){1,2}"
    ),
    
    "аббревиатуры: буквенные": re.compile(r"(?<![А-Яа-яЁё])[БВГДЖЗЙКЛМНПРСТФХЦЧШЩ]{2,5}(?![А-Яа-яЁё])"),
    "римские цифры": re.compile(r"(?<![А-Яа-яЁёA-Za-z])[XVI]{2,}(?![А-Яа-яЁёa-z])"),
    "скобки": re.compile(r"[()\[\]]"),
    "нестандартные кавычки": re.compile(r"[“”„‟‘’‚]"),
    "дефисы и тире": re.compile(r"[‐‑‒–―−]"),
    "многоточие из точек": re.compile(r"\.\.\."),
    "пробелы": re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF\u00AD\u00A0]"),
    "серии знаков": re.compile(
        r"(?!\?!) [!?]{2,}"
        r"| [!?]\."
        r"| \.[!?]"
        r"| (?<![.!?]) \.{2} (?!\.)",
        re.VERBOSE,
    ),
    
    "буква ё": re.compile(r"[ёЁ]"),
    "эмодзи": re.compile(
            r"[\U0001F300-\U0001FAFF"
            r"\U00002700-\U000027BF"
            r"\U0001F1E6-\U0001F1FF]"
    ),
    
    "текстовые смайлики": re.compile(
        r"(?<!\S)"
        r"(?:"
        r"[:;=8][\-^']?[)(DPpOo/\\]"
        r"|"
        r"[)(]{2,}"
        r"|"
        r"\^[_-]\^"
        r"|"
        r"[Tt]_[Tt]"
        r"|"
        r"[Oo]_[Oo]"
        r"|"
        r"[xX][dD]+"
        r")"
        r"(?!\S)"
    ),
    
    "междометия": re.compile(
        r"(?<![А-Яа-яЁё])"
        r"(?:"
        r"ах(?:ах)+|"
        r"ха(?:ха)+|"
        r"хи(?:хи)+|"
        r"уфф+|"
        r"хм+|"
        r"ыы+|"
        r"ухх+|"
        r"эхх+|"
        r"ых+|"
        r"м{2,}|"
        r"мгм|"
        r"гм|"
        r"угу|"
        r"бр{2,}"
        r")"
        r"(?![А-Яа-яЁё])",
        re.IGNORECASE
    )
}

In [101]:
def load_corpus(path = INPUT_PATH):
    
    data = pd.read_csv(path, names = ["id", "text"], **CSV_KWARGS)
    data["text"] = data["text"].fillna("").astype(str)
    
    return data

# получаем количество строк и вхождений каждого типа ненормализованных данных
def type_counts(data):

    rows = []
    total = len(data)
    
    for name, pattern in PATTERNS.items():
        mask = data["text"].str.contains(pattern, regex=True, na=False)

        matches = data["text"].str.count(pattern)
        
        rows.append({
            "тип": name,
            "строк": int(mask.sum()),
            "доля, %": round(100 * mask.sum() / total, 2),
            "вхождений": int(matches.sum()),
        })
        
    return (
        pd.DataFrame(rows)
        .sort_values("строк", ascending=False)
        .reset_index(drop=True)
    )


def examples(data, pattern, limit = 1):
 
    mask = data["text"].str.contains(pattern, regex=True, na=False)
    
    return [
        f"{row.id} | {row.text}"
        for _, row in data.loc[mask].head(limit).iterrows()
    ]


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    
    data = load_corpus()

    print(f"Строк в корпусе: {len(data)}")
    
    lengths = data["text"].str.len()
    words = data["text"].str.split().str.len()
    
    print(f"Длина строки, символов: медиана {lengths.median():.0f}, "
          f"минимум {lengths.min()}, максимум {lengths.max()}")

    print(f"Слов в строке: медиана {words.median():.0f}, "
          f"максимум {words.max()}")

    print(f"Уникальных ID: {data['id'].nunique()}")
    print(f"Дубликатов ID: {data['id'].duplicated().sum()}")
    print(f"Уникальных текстов: {data['text'].nunique()}")
    print(f"Дубликатов текстов: {data['text'].duplicated().sum()}")
    print(
        f"Пустых строк: "
        f"{data['text'].str.strip().eq('').sum()}"
    )

    # статистика по типам
    types = type_counts(data)

    types.to_csv(
        f"{OUT_DIR}/types.csv",
        index=False,
        encoding="utf-8"
    )

    print("\nТипы ненормализованных данных:")
    print(types.to_string(index=False))

    # примеры найденных конструкций
    print("\nПримеры найденных конструкций:")

    examples_path = f"{OUT_DIR}/examples.txt"

    with open(
        examples_path,
        "w",
        encoding="utf-8"
    ) as handle:

        for name, pattern in PATTERNS.items():
            found_examples = examples(data, pattern)

            # не выводим категории, которых в корпусе нет
            if not found_examples:
                continue

            print(f"\n=== {name} ===")

            handle.write(f"=== {name} ===\n")

            for line in found_examples:
                print(line)
                handle.write(f"{line}\n")

            handle.write("\n")

    print(
        f"\nПримеры сохранены в: "
        f"{examples_path}"
    )

In [102]:
if __name__ == "__main__":
    main()

Строк в корпусе: 22200
Длина строки, символов: медиана 85, минимум 10, максимум 723
Слов в строке: медиана 12, максимум 111
Уникальных ID: 22200
Дубликатов ID: 0
Уникальных текстов: 21940
Дубликатов текстов: 260
Пустых строк: 0

Типы ненормализованных данных:
                            тип  строк  доля, %  вхождений
                  дефисы и тире   6789    30.58      12653
                   серии знаков    683     3.08        712
                         скобки    307     1.38        610
          нестандартные кавычки     63     0.28        148
                       инициалы     54     0.24         61
        аббревиатуры: буквенные     31     0.14         33
            многоточие из точек     22     0.10         25
                        буква ё     13     0.06         17
            технические символы      5     0.02          7
                          цифры      4     0.02          8
сокращения: общеупотребительные      2     0.01          3
                     междометия 

* Дополнительная проверка дефисов и тире:

In [103]:
data = load_corpus()

dash_hyphen_chars = "‐‑‒–―−-"

for char in dash_hyphen_chars:
    count = data["text"].str.count(re.escape(char)).sum()
    code = f"U+{ord(char):04X}"
    name = unicodedata.name(char, "UNKNOWN")
    
    print(f"{repr(char)} | {code} | {name} | {count}")


dash_examples = data[
    data["text"].str.contains(char, regex=False, na=False)
]

for _, row in dash_examples.sample(
    
    n=min(5, len(dash_examples)),
    random_state=42
).iterrows():
    
    text = row.text

    for i, current_char in enumerate(text):
        if current_char == char:
            start = max(0, i - 20)
            end = min(len(text), i + 21)
            print(f"  {row.id} | ...{text[start:end]}...")

'‐' | U+2010 | HYPHEN | 0
'‑' | U+2011 | NON-BREAKING HYPHEN | 2243
'‒' | U+2012 | FIGURE DASH | 0
'–' | U+2013 | EN DASH | 10410
'―' | U+2015 | HORIZONTAL BAR | 0
'−' | U+2212 | MINUS SIGN | 0
'-' | U+002D | HYPHEN-MINUS | 517
  021581_RUSLAN | ...- А я и не заметил. К...
  021581_RUSLAN | ...... Дверцы, руль... - Это не та машина, -...
  021581_RUSLAN | ...- Это не та машина, - весело сказал Ариэл...
  021542_RUSLAN | ...- Дядя Гриша! Вам нуж...
  001968_RUSLAN | ...Да, в какой-то мере пока существ...
  005043_RUSLAN | ...дит нам и выговорить-то противно....
  003069_RUSLAN | ...Неожиданно он что-то выкрикнул противн...


* Анализ вхождений разных видов тире и дефиса, показал, что символ U-2013 используется повсеместно (длинное тире), а вот дефисы встречаются двух видов: U+2011 и U+002D, поэтому при нормализации стоит прийти к единообразному варианту.

## Отчет по разведочному анализу корпуса RUSLAN

Корпус содержит 22 200 строк, из которых - 21 940 уникальных текстов и 260 дубликатов. Из ненормализованных данных самый частый тип - дефисы и тире: они встречаются в 30,58 % корпуса. В примерах видны неразрывные дефисы в словах вроде «как‑то», «кое‑что», «чуть‑чуть», которые нужно будет нормализовать. Следующие по численности - серии знаков (712 вхождений). Некоторые из них подсказывают интонацию, но большинство вариантов все же должны сводиться к стандартным знакам. Скобки также встречаются достаточно часто, однако их не стоит критично подвергать полной фильтрации. Нестандартные кавычки следует  нормализовать. Инициалы тоже являются проблемой и попадаются довольно часто. Поскольку расшифровать каждое имя не представляется возможным, а читать по буквам неестественно, то их, скорее всего придется удалить. Многоточие "из точек" в конце реплик, требует нормализации «...» → «…». Буква ё также встречается, и ее нельзя терять, так как она влияет на произношение, а нередко и на смысл. Технические символы: «/», «*», «<…>» - являются шумом. Цифры требуют особого чтения и склонения, однако в сочетаниях вроде «ТУ‑129» произносятся в начальной форме, а значит можно попробовать некоторые из них оставить. Встречаются единичные случаи различных сокращений (географических, должностных и др.) их «расшифровка» так же представляет проблему, поэтому они подвергнутся фильтрации. Латинских символов, эмодзи, текстовых смайликов и невидимых символов в корпусе не нашлось. Отдельная существенная проблема - слова, написанные в верхнем регистре. Они визуально неотличимы от буквенных аббревиатур, и регулярное выражение, настроенное на «все заглавные буквы», начинает ловить обычные слова вроде «ПРИВЕТ», «СТОП», «ВСЕ», что для TTS опасно. Поэтому было принято решение искать только те аббревиатуры, которые полностью состоят из согласных букв, что отсекает слова с гласными и оставляет настоящие аббревиатуры вроде «КПСС», «МТС», «ВМК». Однако под выбранное регулярное выражение не попадут аббревиатуры с гласными, которые тоже читаются по буквам («МГУ», «ЭВМ»), а также аббревиатуры, которые читаются в смешанном виде, например, «ЦСКА», «ГИБДД», и их придется обрабатывать отдельно.

## Фильтр

In [104]:
from sklearn.metrics import f1_score, precision_score, recall_score

DEV_SET_PATH = "data/dev_sentences.csv"

class TextFilter:

    def __init__(self):
        
        self.digits_re = re.compile(r"\d")

        self.latin_re = re.compile(r"[A-Za-z]")

        self.tech_symbols_re = re.compile(r"[*/@+<>\\|^~_#={}]")

        self.designations_re = re.compile(r"[%°$€₽№§©®&]")

        self.abbreviations_re = re.compile(
            r"(?<![А-Яа-яЁё])"
            r"(?:"
            r"т\.\s*[дпек]\.|"
            r"др\.|"
            r"им\.|"

            r"мм|см|дм|км|мг|кг|мл|л|мин|сек|"

            r"руб\.|коп\.|тыс\.|млн\.|млрд\.|"

            r"ул\.|пл\.|просп\.|наб\.|пер\.|бул\.|ш\.|"
            r"тр\.|кв\.|обл\.|р-н|пос\.|дер\.|"

            r"рис\.|табл\.|гл\.|стр\.|см\.|ср\.|"
            r"напр\.|прим\.|разд\.|парагр\.|изд\.|ред\.|сост\.|"

            r"проф\.|акад\.|доц\.|ассист\.|инж\.|техн\.|"
            r"дир\.|зав\.|зам\.|тов\.|гр\.|г-н|г-жа|"

            r"чел\.|экз\.|шт\.|ед\."

            r"|гос\.|гос-во|об-во|орг\.|орг-ция|"
            r"предпр\.|учр\.|стр-во|хоз-во"

            r"|г\.|в\.|вв\.|н\.\s*э\.|ч\."
            r")"
            r"(?![А-Яа-яЁё])",
            re.IGNORECASE
        )

        self.initials_re = re.compile(
            r"(?<![А-Яа-яЁё])"
            r"(?:[А-ЯЁ]\.\s*){1,2}[А-ЯЁ][а-яё]+"
            r"|"
            r"[А-ЯЁ][а-яё]+\s+(?:[А-ЯЁ]\.\s*){1,2}"
        )

        self.abbreviation_caps_re = re.compile(
            r"(?<![А-Яа-яЁё])"
            r"[БВГДЖЗЙКЛМНПРСТФХЦЧШЩ]{2,5}"
            r"(?![А-Яа-яЁё])"
        )

        self.quotes_re = re.compile(r"[“”„‟‘’‚]")

        self.repeated_punctuation_re = re.compile(
            r"""
            (?!\?!) [!?]{2,}          
            | (?<![.!?]) \.{2} (?!\.) 
            | [!?]\.                  
            | \.[!?]                  
            """,
            re.VERBOSE,
        )

        self.invisible_re = re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF\u00AD\u00A0]")

        self.emoji_re = re.compile(
            r"[\U0001F300-\U0001FAFF"
            r"\U00002700-\U000027BF"
            r"\U0001F1E6-\U0001F1FF]"
        )

        self.text_smiley_re = re.compile(
            r"(?<!\S)"
            r"(?:"
            r"[:;=8][\-^']?[)(DPpOo/\\]"
            r"|"
            r"[)(]{2,}"
            r"|"
            r"\^[_-]\^"
            r"|"
            r"[Tt]_[Tt]"
            r"|"
            r"[Oo]_[Oo]"
            r"|"
            r"[xX][dD]+"
            r")"
            r"(?!\S)"
        )

        self.interjection_re = re.compile(
            r"(?<![А-Яа-яЁё])"
            r"(?:"
            r"ах(?:ах)+|"
            r"ха(?:ха)+|"
            r"хи(?:хи)+|"
            r"уфф+|"
            r"ухх+|"
            r"эхх+|"
            r"ыхх+|"
            r"хм+|"
            r"ыы+|"
            r"м{2,}|"
            r"мгм|"
            r"гм|"
            r"угу|"
            r"бр{2,})",
            re.IGNORECASE
        )
        
        self.space_before_punctuation_re = re.compile(r"\s+([,.!?;:])")

        self.space_after_punctuation_re = re.compile(r"([,;:])(?=\S)")
        

    def filter(self, text):

        if not isinstance(text, str) or not text.strip():
            return 0

        text = unicodedata.normalize("NFC", text)

        checks = [
            self.digits_re,
            self.latin_re,
            self.tech_symbols_re,
            self.designations_re,
            self.abbreviations_re,
            self.initials_re,
            self.abbreviation_caps_re,
            self.quotes_re,
            self.repeated_punctuation_re,
            self.invisible_re,
            self.emoji_re,
            self.text_smiley_re,
            self.interjection_re,
            self.space_before_punctuation_re,
            self.space_after_punctuation_re
        ]

        for pattern in checks:
            if pattern.search(text):
                return 0

        return 1

if __name__ == "__main__":
    
    textfilter = TextFilter()

    dev_files = pd.read_csv(
        DEV_SET_PATH,
        sep="|",
        encoding="utf-8",
        quoting=csv.QUOTE_NONE,
        header=0
    )

    dev_files["predicted"] = dev_files["text"].apply(
        textfilter.filter
    )

    prc = precision_score(
        dev_files["is_normalized"],
        dev_files["predicted"]
    )

    rec = recall_score(
        dev_files["is_normalized"],
        dev_files["predicted"]
    )

    f1 = f1_score(
        dev_files["is_normalized"],
        dev_files["predicted"]
    )

    print(
        f"F1 Score is {f1:.4f}, "
        f"Precision is {prc:.4f}, "
        f"Recall is {rec:.4f}"
    )

F1 Score is 0.9562, Precision is 0.9281, Recall is 0.9861


## Нормализатор

In [105]:
class TextNormalizer:
    
    def __init__(self):

        self.multi_space_re = re.compile(r" {2,}") # два или больше стандартных пробелов подряд

        # дефисы
        self.hyphen_map = str.maketrans({"‑": "-"})

        # технические символы
        self.tech_symbols_re = re.compile(r"[*/@+<>\\|^~_#={}%°$€₽№§©®]")

        # кавычки
        self.quote_map = str.maketrans({
            "“": '"',
            "”": '"',
            "„": '"',
            "»": '"',
            "«": '"',
            "’": "'"
        })

        # рандомные точки внутри предложения
        self.random_dot_re = re.compile(r"(?<=[А-Яа-яЁё])\s*\.(?=\s+[а-яё])")

        # .,. >> …
        self.comma_dot_comma_re = re.compile(r"\.,\.")

        # серии знаков препинания
        self.punctuation_series_re = re.compile(
            r"""
            \?!\.*          
            | [!?]{3,}      
            | [!?]\.+       
            | \.{2,}[!?]    
            | \.{2,}      
            """,
            re.VERBOSE,
        )

        # пробелы вокруг пунктуации
        self.space_before_punctuation_re = re.compile(r"\s+([,.!?;:])")

        self.space_after_punctuation_re = re.compile(r"([,;:])(?=\S)")

        # однозначные общеупотребительные сокращения
        self.abbreviations = {
            "т.е.": "то есть",
            "т. е.": "то есть",
            "т.к.": "так как",
            "т. к.": "так как",
            "т.д.": "так далее",
            "т. д.": "так далее",
            "т.п.": "тому подобное",
            "т. п.": "тому подобное",
        }

        self.abbreviation_re = re.compile(
            r"(?<![А-Яа-яЁё])"
            r"(т\.\s*е\.|т\.\s*к\.|т\.\s*д\.|т\.\s*п\.)"
            r"(?![А-Яа-яЁё])",
            re.IGNORECASE
        )

        # числа в буквенно-численных обозначениях
        self.alphanumeric_number_re = re.compile(r"\b([А-ЯЁа-яё]+)-(\d+)\b")  

    # расшифровка цифр
    def number_to_words(self, number):
        
        number = int(number)
    
        units = {
            0: "ноль",
            1: "один",
            2: "два",
            3: "три",
            4: "четыре",
            5: "пять",
            6: "шесть",
            7: "семь",
            8: "восемь",
            9: "девять",
        }
    
        teens = {
            10: "десять",
            11: "одиннадцать",
            12: "двенадцать",
            13: "тринадцать",
            14: "четырнадцать",
            15: "пятнадцать",
            16: "шестнадцать",
            17: "семнадцать",
            18: "восемнадцать",
            19: "девятнадцать",
        }
    
        tens = {
            20: "двадцать",
            30: "тридцать",
            40: "сорок",
            50: "пятьдесят",
            60: "шестьдесят",
            70: "семьдесят",
            80: "восемьдесят",
            90: "девяносто",
        }
    
        hundreds = {
            100: "сто",
            200: "двести",
            300: "триста",
            400: "четыреста",
            500: "пятьсот",
        }
    
        if number < 10:
            return units[number]
    
        if number < 20:
            return teens[number]
    
        if number < 100:
            return tens[number // 10 * 10] + (
                " " + units[number % 10]
                if number % 10
                else ""
            )
    
        if number <= 500:
            hundred = number // 100 * 100
            remainder = number % 100
    
            if remainder == 0:
                return hundreds[hundred]
    
            return hundreds[hundred] + " " + self.number_to_words(remainder)
    
        return str(number)


    def normalize(self, text):

        if not isinstance(text, str) or not text:
            return ""

        text = unicodedata.normalize("NFC", text)

        text = text.translate(self.hyphen_map)

        def replace_alphanumeric_number(match):
            letters = match.group(1)
            number = match.group(2)

            return letters + " " + self.number_to_words(number)

        text = self.alphanumeric_number_re.sub(
            replace_alphanumeric_number,
            text
        )

        def replace_abbreviation(match):
            abbreviation = match.group(0).lower()

            abbreviation = abbreviation.replace(" ", "")

            return self.abbreviations[abbreviation]

        def replace_punctuation(match):
            value = match.group(0)
        
            if "?" in value and "!" in value:
                return "?!"
        
            if "?" in value:
                return "?"
        
            if "!" in value:
                return "!"
        
            return "…"       

        text = self.abbreviation_re.sub(
            replace_abbreviation,
            text
        )

        text = text.translate(self.quote_map)

        text = self.tech_symbols_re.sub(" ", text)

        text = self.space_before_punctuation_re.sub(r"\1", text)

        text = self.punctuation_series_re.sub(replace_punctuation, text)

        text = self.comma_dot_comma_re.sub("…", text)

        text = self.random_dot_re.sub("", text)

        text = self.space_after_punctuation_re.sub(r"\1 ", text)

        text = self.multi_space_re.sub(" ", text)

        text = unicodedata.normalize("NFC", text)

        return text.strip()

* Проверим работу нормализатора:

In [106]:
data = load_corpus()
normalizer = TextNormalizer()

def replace_alphanumeric_number(match):
    letters = match.group(1)
    number = match.group(2)

    return letters + " " + normalizer.number_to_words(number)

def replace_abbreviation(match):
    abbreviation = match.group(0).lower()
    abbreviation = abbreviation.replace(" ", "")

    return normalizer.abbreviations[abbreviation]

def replace_punctuation(match):
    value = match.group(0)

    if "?" in value and "!" in value:
        return "?!"

    if "?" in value:
        return "?"

    if "!" in value:
        return "!"

    return "…"

def normalize_hyphens(text):
    
    return text.translate(normalizer.hyphen_map)

def normalize_alphanumeric(text):
    
    return normalizer.alphanumeric_number_re.sub(replace_alphanumeric_number, text)

def normalize_abbreviations(text):
    
    return normalizer.abbreviation_re.sub(replace_abbreviation, text)

def normalize_quotes(text):
    
    return text.translate(normalizer.quote_map)

def normalize_technical_symbols(text):
    
    return normalizer.tech_symbols_re.sub(" ", text)

def normalize_punctuation(text):
    
    return normalizer.punctuation_series_re.sub(replace_punctuation, text)

def normalize_comma_dot_comma(text):
    
    return normalizer.comma_dot_comma_re.sub("…", text)

def normalize_random_dots(text):
    
    return normalizer.random_dot_re.sub("", text)

def normalize_spaces_before(text):
    
    return normalizer.space_before_punctuation_re.sub(r"\1", text)

def normalize_spaces_after(text):
    
    return normalizer.space_after_punctuation_re.sub(r"\1 ", text)

def normalize_multiple_spaces(text):
    
    return normalizer.multi_space_re.sub(" ", text)


current = data["text"].copy()
changed_indices = set()

# дефисы
previous = current.copy()
current = current.apply(normalize_hyphens)

changed = previous != current
changed_indices.update(data.index[changed])

print("=== Дефисы ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# буквенно-численные обозначения
previous = current.copy()
current = current.apply(normalize_alphanumeric)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Буквенно-численные обозначения ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# сокращения
previous = current.copy()
current = current.apply(normalize_abbreviations)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Сокращения ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# кавычки
previous = current.copy()
current = current.apply(normalize_quotes)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Кавычки ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# технические символы
previous = current.copy()
current = current.apply(normalize_technical_symbols)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Технические символы ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# пробелы перед пунктуацией
previous = current.copy()
current = current.apply(normalize_spaces_before)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Пробелы перед пунктуацией ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# серии знаков препинания
previous = current.copy()
current = current.apply(normalize_punctuation)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Серии знаков препинания ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# .,. → …
previous = current.copy()
current = current.apply(normalize_comma_dot_comma)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== .,. → … ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# случайные точки внутри предложения
previous = current.copy()
current = current.apply(normalize_random_dots)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Случайные точки внутри предложения ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# пробелы после пунктуации
previous = current.copy()
current = current.apply(normalize_spaces_after)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Пробелы после пунктуации ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

# повторяющиеся пробелы
previous = current.copy()
current = current.apply(normalize_multiple_spaces)

changed = previous != current
changed_indices.update(data.index[changed])

print("\n=== Повторяющиеся пробелы ===")
print(f"Изменено строк: {changed.sum()}")

if changed.sum() > 0:
    idx = data.index[changed][0]
    print(f"Было:  {previous.loc[idx]}")
    print(f"Стало: {current.loc[idx]}")

print("\n" + "=" * 50)
print(f"ВСЕГО изменено уникальных строк: {len(changed_indices)}")
print(f"Из {len(data)} строк корпуса")
print(f"Доля: {100 * len(changed_indices) / len(data):.2f}%")

=== Дефисы ===
Изменено строк: 2006
Было:  Моя жена говорит, если бы ты оделся как следует… Между прочим, его жена звонит как‑то раз… Стоп! 
Стало: Моя жена говорит, если бы ты оделся как следует… Между прочим, его жена звонит как-то раз… Стоп! 

=== Буквенно-численные обозначения ===
Изменено строк: 4
Было:  Брат разъезжал по отдаленным лагерным точкам. Ему предоставили казенную машину «ГАЗ-61». 
Стало: Брат разъезжал по отдаленным лагерным точкам. Ему предоставили казенную машину «ГАЗ шестьдесят один». 

=== Сокращения ===
Изменено строк: 2
Было:  Они – моя неврастения, злость, апломб, беспечность. И т. д. И самая кровавая война – бой призраков. 
Стало: Они – моя неврастения, злость, апломб, беспечность. И так далее И самая кровавая война – бой призраков. 

=== Кавычки ===
Изменено строк: 1892
Было:  Мечтаем получить от вас рецензию.”
Стало: Мечтаем получить от вас рецензию."

=== Технические символы ===
Изменено строк: 5
Было:  Я на букву О /библиография к Окуджаве/.
Стало: Я на бук

## Полная обработка

In [107]:
INPUT_PATH = "data/RUSLAN/metadata_RUSLAN_22200.csv"
OUTPUT_PATH = "data/metadata_RUSLAN_22200_normalized.csv"

CSV_KWARGS = {
    "sep": "|",
    "quoting": csv.QUOTE_NONE
}

if __name__ == "__main__":
    text_filter = TextFilter()
    normalizer = TextNormalizer()

    raw = pd.read_csv(
        INPUT_PATH,
        names=["id", "raw"],
        **CSV_KWARGS
    )

    raw["nrm"] = raw["raw"].apply(normalizer.normalize)

    clean = raw[
        raw["nrm"].apply(text_filter.filter) == 1
    ]

    print(
        f"Num rows before cleaning: {len(raw)}; "
        f"after cleaning: {len(clean)}"
    )

    clean[["id", "raw", "nrm"]].to_csv(
        OUTPUT_PATH,
        index=False,
        header=False,
        **CSV_KWARGS
    )

Num rows before cleaning: 22200; after cleaning: 22060
